# YOLO pose and movement pipeline

## My contribution: pose extraction for Octagon Analytics

This notebook is the computer-vision branch of the group project. Its job is to convert a fight video into reproducible, per-frame fighter pose and position data that the team can later explore alongside the existing event and fighter datasets. It **does not replace the group’s tabular fight-outcome work**, and a pose model by itself does not predict who wins a fight.

### Notebook map and suggested run order

1. **Setup** — install dependencies once in the selected Jupyter kernel.
2. **Sanity check** — run the bus-image example below to confirm the pretrained model returns a human skeleton.
3. **Select a bout** — choose one uniquely matching event row and its authorized local video.
4. **Extract poses** — track people through the video, identify the two fighter track IDs, and save their normalized keypoints.
5. **Label movement windows** — add human-reviewed time ranges such as `clinch` or `takedown_attempt`; only then can a separate movement classifier be trained.

### Handoff artifact for the group

The concrete output is `data/derived/<video-name>_poses.csv`. Each row represents one detected person in one video frame and includes a temporary video track ID, normalized bounding-box position, and 17 COCO keypoints. To make it useful for the rest of the team, record the selected event/fighters, video source/permission, frame rate, two fighter track IDs, and any excluded rounds or camera cuts.

**Fill-in checkpoints:** [ ] video selected  [ ] skeletons visually checked  [ ] fighter IDs recorded  [ ] pose CSV saved  [ ] movement labels reviewed

*Note: we will use YOLO's pretrained model (hence the pt file) for doing generic testing since creating a model at this time was much heftier than expected.

In [ ]:
#if need be, run this once on your appropriate kernel
%pip install ultralytics opencv-python

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.4 MB 669.8 kB/s eta 0:00:02
   ----------------------------- ---------- 1.0/1.4 MB 1.0 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.4 MB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 1.1 MB/s  0:00:01
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/44.0 MB 1.7 MB/s eta 0:00:26
    --------------------------------------- 0.8/44.0 MB 1.8 MB/s eta 0:00:25
   - -------------------------------------- 1.3/44.0 MB 1.9 MB/s eta 0:00:23
   - -------------------------------------- 1.8/44


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Checkpoint 1 — verify the pretrained pose model

This is deliberately a simple image test. If the boxes and skeletons look sensible here, any later error is more likely to be caused by the fight video, tracking, or labels rather than the installation. The pretrained model uses the standard COCO human skeleton; the next markdown cell lists its joint order.


## Skeleton layout
From indices 0 to 16: <br>
Nose <br>
Left Eye <br>
Right Eye <br>
Left Ear <br>
Right Ear <br>
Left Shoulder <br>
Right Shoulder <br>
Left Elbow <br>
Right Elbow <br>
Left Wrist <br>
Right Wrist <br>
Left Hip <br>
Right Hip <br>
Left Knee <br>
Right Knee <br>
Left Ankle <br>
Right Ankle <br>

### Sample Algorithm Test
This will test a sample picture, and under the predict folder, the resulting image with individuals are appropiately marked and joints are drawn. Run this after installing the correct dependencies to ensure the model will correctly run later on.

In [2]:
from ultralytics import YOLO

# 1. Load a pre-trained pose estimation model.
# The 'n' stands for nano, the smallest and fastest version.
model = YOLO("yolo26n-pose.pt")

# 2. Run prediction on an image (or a video, webcam, URL, etc.).
#    - 'show=True' displays the result in a window with bounding boxes and skeletons.
#    - 'save=True' saves the annotated image to the 'runs/pose/predict' folder.
results = model.predict(
    source="https://ultralytics.com/images/bus.jpg", 
    show=True,
    save=True,
    conf=0.5,  # Confidence threshold
    exist_ok=True
)

# 3. You can also access the raw keypoint data for custom processing.
for result in results:
    # result.keypoints.xy gives pixel coordinates (N, 17, 2)
    # result.keypoints.xyn gives normalized coordinates
    print(f"Detected {len(result.keypoints)} person(s).")
    if len(result.keypoints) > 0:
        print("First person's keypoints (x, y):")
        print(result.keypoints.xy[0])


Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 e:\yosoy\OneDrive\Documents\Cal Poly\DATA 3301\group_project\Octagon_Analytics\bus.jpg: 640x480 3 persons, 175.5ms
Speed: 17.1ms preprocess, 175.5ms inference, 18.4ms postprocess per image at shape (1, 3, 640, 480)
Results saved to E:\yosoy\OneDrive\Documents\Cal Poly\DATA 3301\group_project\Octagon_Analytics\runs\pose\predict
Detected 3 person(s).
First person's keypoints (x, y):
tensor([[141.2245, 445.2361],
        [147.9574, 436.2876],
        [131.2376, 435.7838],
        [151.6131, 439.8766],
        [108.4220, 438.5798],
        [159.4579, 491.2240],
        [ 87.0343, 491.9364],
        [184.5287, 567.8741],
        [117.1156, 566.6171],
        [162.1403, 567.9654],
        [164.4060, 551.7612],
        [152.6256, 641.3663],
        [ 97.8333, 642.9088],
        [167.8434, 751.9741],
        [ 85.9676, 753.4797],
        [189.3479, 859.7717],
        [ 72.8797, 862.5049]])


## From one fight video to movement data

**What YOLO provides:** a bounding box and 17 body keypoints for every detected person in each frame. It does *not* predict a punch, takedown, or winner. We will first use its pretrained pose model to turn a legally obtained fight video into a table of tracked fighter positions. A second, small time-series model can learn movement labels from that table.

Use a full fight video (which can be a time-consuming task) to use with the model. This is the part that can be drastic for performance, should every event and fight be monitored in real time. For the feasibility and scope of this project, I'm planning to start it off testing with one video (I got a little behind so at least 1 will be enough for the preliminary).

For the first pass, choose **one** row in `events_raw.csv`, save its matching video under `data/videos/`, and manually record where each round starts. Do not try to train a model yet: inspect the tracked skeletons first.

Since the video should be stored locally for performance reasons, create a gitignore for .mp4 files (assuming the fight video is in that format).

In [3]:
from pathlib import Path

import pandas as pd

events = pd.read_csv("events_raw.csv")

# Pick a bout that you have a matching, authorized video for. This example is
# the first bout in events_raw.csv; change all three fields together if needed.
BOUT = {
    "event_name": "UFC Fight Night: Grasso vs. Shevchenko 2",
    "fighter1": "Jack Della Maddalena",
    "fighter2": "Kevin Holland",
}
match = events.loc[
    (events["event_name"] == BOUT["event_name"])
    & (events["fighter1"] == BOUT["fighter1"])
    & (events["fighter2"] == BOUT["fighter2"])
]
assert len(match) == 1, "Choose a unique event/fighter1/fighter2 combination."
display(match[["event_name", "event_date", "fighter1", "fighter2", "outcome", "Method", "Round", "Time"]])

# This video is intentionally not in Git. Create data/videos locally and put an
# authorized copy there. Keeping the file name tied to the BOUT prevents mix-ups.
VIDEO_PATH = Path("data/videos/welterweight.mp4")
if not VIDEO_PATH.exists():
    print(f"Add an authorized video at: {VIDEO_PATH.resolve()}")


,event_name,event_date,fighter1,fighter2,outcome,Method,Round,Time
1,UFC Fight Night: Grasso vs. Shevchenko 2,September 16 2023,Jack Della Maddalena,Kevin Holland,fighter1,S-DEC,3,5:00


### Extract and track the two fighters

`track()` assigns a temporary ID to each detected person. The referee and people at ringside may also be detected, so run the cell once, inspect `track_summary`, and enter the two IDs that belong to the fighters. IDs are only reliable within this one video; they are not fighter IDs from the CSV.

The saved CSV uses normalized coordinates (`0` to `1`), so features remain comparable if future videos have different resolutions.

In [4]:
import cv2
import numpy as np

assert VIDEO_PATH.exists(), "Set VIDEO_PATH to an authorized fight video before running this cell."

capture = cv2.VideoCapture(str(VIDEO_PATH))
fps = capture.get(cv2.CAP_PROP_FPS)
capture.release()
assert fps > 0, "OpenCV could not read the video's frame rate."

# Track only the COCO 'person' class. stream=True avoids holding the whole video in memory.
pose_model = YOLO("yolo26n-pose.pt")
rows = []
for frame, result in enumerate(pose_model.track(
    source=str(VIDEO_PATH), stream=True, persist=True, classes=[0],
    tracker="bytetrack.yaml", conf=0.25, verbose=False
)):
    if result.boxes.id is None or result.keypoints is None:
        continue

    # The box, ID, and keypoint arrays are aligned by detection.
    track_ids = result.boxes.id.int().cpu().tolist()
    boxes = result.boxes.xywhn.cpu().numpy()       # normalized center-x, center-y, width, height
    keypoints = result.keypoints.xyn.cpu().numpy() # normalized (17, 2) COCO skeleton
    for track_id, (cx, cy, w, h), skeleton in zip(track_ids, boxes, keypoints):
        row = {
            "frame": frame, "time_s": frame / fps, "track_id": track_id,
            "cx": cx, "cy": cy, "box_w": w, "box_h": h,
        }
        row.update({f"kp_{joint}_{axis}": value
                    for joint, point in enumerate(skeleton)
                    for axis, value in zip(("x", "y"), point)})
        rows.append(row)

all_poses = pd.DataFrame(rows)
track_summary = (all_poses.groupby("track_id")
                .agg(frames=("frame", "size"), start_s=("time_s", "min"), end_s=("time_s", "max"))
                .sort_values("frames", ascending=False))
display(track_summary)

# After viewing a short annotated preview, replace these with the two fighter IDs
# from track_summary. Do not assume the two longest tracks are always the fighters.
FIGHTER_TRACK_IDS = {BOUT["fighter1"]: None, BOUT["fighter2"]: None}

output_path = Path("data/derived") / f"{VIDEO_PATH.stem}_poses.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
all_poses.to_csv(output_path, index=False)
print(f"Saved {len(all_poses):,} detected poses to {output_path}")


requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 16.6 MB/s  0:00:00

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip

requirements: AutoUpdate success  4.9s
WARNING requirements: Restart runtime or rerun command for updates to take effect



,frames,start_s,end_s
track_id,,,
13864,910,438.333333,468.666667
11156,734,341.633333,366.933333
4959,644,147.933333,169.633333
14467,630,454.466667,475.666667
12822,556,412.833333,432.300000
...,...,...,...
8748,1,265.266667,265.266667
16418,1,524.066667,524.066667
16402,1,523.666667,523.666667


Saved 84,391 detected poses to data\derived\welterweight_poses.csv


### Turn poses into a movement prediction

After assigning the two track IDs, calculate features in short windows (for example, one second): fighter-to-fighter distance, each fighter's centre velocity, closing/separating speed, wrist velocity, hip/shoulder orientation, and stance width. Smooth or interpolate only short gaps; occlusions during clinches and ground fighting are expected.

Create a separate `data/annotations/movement_labels.csv` with `video_id,start_s,end_s,label`, where label might be `distance_closing`, `distance_opening`, `striking_exchange`, `clinch`, `takedown_attempt`, or `ground`. Those labels come from a person watching the video (or from a trustworthy timestamped play-by-play source), not from the aggregate `Str`, `Td`, and `Kd` columns. Split train/validation/test by **video or bout**, never randomly by frame, to avoid leakage from near-identical adjacent frames.

The fight-level CSV can still be useful as a final sanity check: predictions across a bout should be plausible given its total knockdowns/strikes/takedowns, but those totals are not frame-level targets.

### Do you need to train YOLO?

Probably not for the first video. Run the pretrained `yolo26n-pose.pt` first. Fine-tune it only if skeletons consistently fail on your octagon footage (camera angle, occlusion, gloves, or ground positions). YOLO pose training requires one label file per image with a person box and all 17 keypoints/visibility values; movement categories alone are *not* YOLO pose labels. Annotate sampled frames in a pose-capable tool, then use a dataset YAML and the guarded sketch below.

In [6]:
# Expected pose dataset layout (not created by this notebook):
# data/fighter_pose/images/{train,val}/frame_001.jpg
# data/fighter_pose/labels/{train,val}/frame_001.txt
# data/fighter_pose/dataset.yaml
#
# dataset.yaml must declare one class plus COCO-compatible keypoints, e.g.:
# path: data/fighter_pose
# train: images/train
# val: images/val
# names: {0: fighter}
# kpt_shape: [17, 3]   # x, y, visibility for each joint
# flip_idx: [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]

RUN_POSE_FINETUNING = False  # Change only after labels have been checked.
if RUN_POSE_FINETUNING:
    finetune_model = YOLO("yolo26n-pose.pt")  # transfer COCO pose knowledge
    train_results = finetune_model.train(
        data="data/fighter_pose/dataset.yaml",
        epochs=50, imgsz=960, patience=10,
        project="runs/fighter_pose", name="v1",
    )
    # Evaluate by held-out BOUT, including hard clinch/ground frames, not just mAP.
